# Band-Wise Churn Analysis
**Purpose:** Churn rates by membership band, reasons for churn per band, and statistical tests confirming band as a significant predictor.

In [0]:
%run ./NB01_Config_and_Setup

In [0]:
billings = load_table(BILLINGS_TABLE, BILLINGS_PATH)
billings = parse_dates(billings, ['Prospect_Renewal_Date','Closed_Date'])
billings['days_to_close'] = (billings['Closed_Date'] - billings['Prospect_Renewal_Date']).dt.days
conditions = [
    (billings['Prospect_Outcome']=='Churned') & (billings['Closed_Date'].isna()),
    (billings['Prospect_Outcome']=='Churned') & (billings['days_to_close'] > CHURN_DAYS_THRESHOLD),
    (billings['Prospect_Outcome']=='Churned') & (billings['days_to_close'] < 0),
    billings['Prospect_Outcome']=='Won',
    billings['Prospect_Outcome']=='Open',
]
billings['Churn_Label'] = np.select(conditions,
    ['Churned','Churned','Pre_Churned','Won','Open'], default='Churned')
model_df = billings[billings['Churn_Label'].isin(['Won','Churned'])].copy()
model_df['Target']       = (model_df['Churn_Label']=='Churned').astype(int)
model_df['Renewal_Year'] = model_df['Prospect_Renewal_Date'].dt.year
mdf = model_df[model_df['Renewal_Year'].isin(TARGET_YEARS)].copy()
print(f"[NB05] {len(mdf):,} records ready")


## 1 · Band Churn Rate Summary

In [0]:
section("Band Churn Rate Summary")
band_summary = mdf.groupby('Band').agg(
    Total=('Target','count'), Churned=('Target','sum')
).reset_index()
band_summary['Retained']   = band_summary['Total'] - band_summary['Churned']
band_summary['Churn_Rate'] = (band_summary['Churned']/band_summary['Total']*100).round(2)
band_summary = band_summary.sort_values('Churn_Rate', ascending=False)
display_df(band_summary, "Band Churn Summary")


In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Churn by Membership Band", fontsize=14)

# Left: churn rate
colors = [rate_color(r) for r in band_summary['Churn_Rate']]
axes[0].barh(band_summary['Band'], band_summary['Churn_Rate'], color=colors, edgecolor='#0a0e1a')
for i, (_, row) in enumerate(band_summary.iterrows()):
    axes[0].text(row['Churn_Rate']+0.1, i,
                 f"{row['Churn_Rate']:.1f}%", va='center', fontsize=9)
axes[0].axvline(mdf['Target'].mean()*100, color='white', linestyle='--',
                linewidth=1, alpha=0.5, label=f"Overall avg {mdf['Target'].mean()*100:.1f}%")
axes[0].set_title("Churn Rate by Band"); axes[0].set_xlabel("Churn Rate (%)"); axes[0].legend()

# Right: stacked volume
axes[1].barh(band_summary['Band'], band_summary['Churned'],
             color=PALETTE['danger'], label='Churned', edgecolor='#0a0e1a', alpha=0.85)
axes[1].barh(band_summary['Band'], band_summary['Retained'],
             left=band_summary['Churned'], color=PALETTE['green'],
             label='Retained', edgecolor='#0a0e1a', alpha=0.5)
axes[1].set_title("Volume: Churned vs Retained"); axes[1].set_xlabel("Accounts"); axes[1].legend()

plt.tight_layout(); plt.show()


## 2 · Band Churn Rate — Year-over-Year

In [0]:
section("Band Churn Rate — Year over Year")
band_yr = mdf.groupby(['Band','Renewal_Year']).agg(
    Total=('Target','count'), Churned=('Target','sum')
).reset_index()
band_yr['Churn_Rate'] = (band_yr['Churned']/band_yr['Total']*100).round(2)

top_bands = band_summary.head(6)['Band'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("Band Churn Rate by Year (Top 6 Bands)", fontsize=14)
for i, band in enumerate(top_bands):
    ax = axes[i//3][i%3]
    bdata = band_yr[band_yr['Band']==band]
    bars = ax.bar(bdata['Renewal_Year'].astype(str), bdata['Churn_Rate'],
                  color=[year_color(y) for y in bdata['Renewal_Year']],
                  edgecolor='#0a0e1a', width=0.5)
    for bar, rate in zip(bars, bdata['Churn_Rate']):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                f"{rate:.1f}%", ha='center', va='bottom', fontsize=9)
    ax.set_title(band); ax.set_ylabel("Churn Rate (%)")
plt.tight_layout(); plt.show()


## 3 · Churn Reasons by Band

In [0]:
section("Churn Reasons by Band")
churn_only = mdf[mdf['Target']==1].copy()
top_reasons = churn_only['Prospect_Status'].value_counts().head(8).index.tolist()

band_reasons = churn_only[churn_only['Prospect_Status'].isin(top_reasons)]    .groupby(['Band','Prospect_Status']).size().reset_index(name='Count')

band_reasons_pivot = band_reasons.pivot_table(
    index='Band', columns='Prospect_Status', values='Count', fill_value=0)
display_df(band_reasons_pivot.reset_index(), "Churn Reasons by Band (count)")


In [0]:
fig, ax = plt.subplots(figsize=(18, 8))
reason_colors = [PALETTE['danger'],'#dc2626','#b91c1c',PALETTE['amber'],
                 '#d97706',PALETTE['blue'],'#2563eb',PALETTE['green']]
band_reasons_pivot.plot(kind='bar', stacked=True, ax=ax,
                        color=reason_colors[:len(band_reasons_pivot.columns)],
                        edgecolor='#0a0e1a', width=0.7)
ax.set_title("Churn Volume by Band and Reason", fontsize=13)
ax.set_xlabel("Membership Band"); ax.set_ylabel("Churned Accounts")
ax.tick_params(axis='x', rotation=30)
ax.legend(bbox_to_anchor=(1.01,1), loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()


In [0]:
# Top reason per band
top_reason_band = churn_only.groupby(['Band','Prospect_Status'])    .size().reset_index(name='Count')    .sort_values(['Band','Count'], ascending=[True,False])    .drop_duplicates('Band')
top_reason_band.columns = ['Band','Top Reason','Count']
display_df(top_reason_band, "Top Churn Reason per Band")


## 4 · Statistical Tests — Does Band Predict Churn?

In [0]:
section("Chi-Square Test: Band vs Churn")
contingency = pd.crosstab(mdf['Band'], mdf['Target'])
chi2, p, dof, expected = chi2_contingency(contingency)
print(f"Chi-square statistic : {chi2:.2f}")
print(f"Degrees of freedom   : {dof}")
print(f"p-value              : {p:.2e}")
if p < 0.05:
    print("  RESULT: Band is a STATISTICALLY SIGNIFICANT predictor of churn (p < 0.05).")
else:
    print("  RESULT: Band is NOT a significant predictor.")

# Cramers V (effect size)
n = contingency.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape)-1)))
print(f"  Cramér's V (effect size) : {cramers_v:.4f}")
print(f"  Interpretation: {'Strong' if cramers_v>0.3 else 'Moderate' if cramers_v>0.1 else 'Weak'} association")
